# **Data Loading and Cleaning**

## MetroPT-3 Air Compressor Dataset (Porto, Portugal)

In [1]:
import pandas as pd
import numpy as np

In [2]:
metro = pd.read_csv('../data/raw/MetroPT-3 Train Dataset/MetroPT3(AirCompressor).csv')
metro.head()

,Unnamed: 0,timestamp,TP2,TP3,H1,DV_pressure,Reservoirs,Oil_temperature,Motor_current,COMP,DV_eletric,Towers,MPG,LPS,Pressure_switch,Oil_level,Caudal_impulses
0,0,2020-02-01 00:00:00,-0.012,9.358,9.340,-0.024,9.358,53.600,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
1,10,2020-02-01 00:00:10,-0.014,9.348,9.332,-0.022,9.348,53.675,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
2,20,2020-02-01 00:00:19,-0.012,9.338,9.322,-0.022,9.338,53.600,0.0425,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
3,30,2020-02-01 00:00:29,-0.012,9.328,9.312,-0.022,9.328,53.425,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
4,40,2020-02-01 00:00:39,-0.012,9.318,9.302,-0.022,9.318,53.475,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0


### MetroPT variable description

The MetroPT-3 dataset contains sensor readings from an air compressor on a metro train in Porto, Portugal. The sensors record the following:

- **TP2** - Pressure on the compressor (bar)
- **TP3** - Pressure at the pneumatic panel (bar)
- **H1** - Pressure drop at the cyclonic separator filter (bar)
- **DV_pressure** - Pressure drop when towers discharge air dryers (bar); zero indicates compressor operating under load
- **Reservoirs** - Downstream pressure of the reservoirs (bar); should be close to TP3
- **Oil_temperature** - Oil temperature on the compressor (°C)
- **Motor_current** - Current of one phase of the three-phase motor (A); it presents values close to 0A - when it turns off, 4A - when working offloaded, 7A - when working under load, and 9A - when it starts working
- **COMP** - Air intake valve; active when compressor is OFF or offloaded
- **DV_eletric** - Compressor outlet valve; active under load, inactive when off or offloaded
- **Towers** - Air drying tower selector; 0 = tower one active, 1 = tower two active
- **MPG** - Starts compressor under load when pressure drops below 8.2 bar
- **LPS** - Low pressure alarm; activates when pressure drops below 7 bar
- **Pressure_switch** - Detects discharge in air-drying towers
- **Oil_level** - Active when oil is below expected levels
- **Caudal_impulses** - Counts air flow pulses from APU to reservoirs

*Source: MetroPT-3 dataset documentation, UCI Machine Learning Repository*

In [3]:
# Fix timestamp
metro['timestamp'] = pd.to_datetime(metro['timestamp'])

# Drop useless column
metro = metro.drop(columns=['Unnamed: 0'])

metro.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1516948 entries, 0 to 1516947
Data columns (total 16 columns):
 #   Column           Non-Null Count    Dtype         
---  ------           --------------    -----         
 0   timestamp        1516948 non-null  datetime64[ns]
 1   TP2              1516948 non-null  float64       
 2   TP3              1516948 non-null  float64       
 3   H1               1516948 non-null  float64       
 4   DV_pressure      1516948 non-null  float64       
 5   Reservoirs       1516948 non-null  float64       
 6   Oil_temperature  1516948 non-null  float64       
 7   Motor_current    1516948 non-null  float64       
 8   COMP             1516948 non-null  float64       
 9   DV_eletric       1516948 non-null  float64       
 10  Towers           1516948 non-null  float64       
 11  MPG              1516948 non-null  float64       
 12  LPS              1516948 non-null  float64       
 13  Pressure_switch  1516948 non-null  float64       
 14  Oi

### Initial Cleaning

The raw dataset contained an unnecessary auto-generated index column (`Unnamed: 0`) which was removed. The `timestamp` column was converted from string to `datetime64` format to enable time-based analysis.

In [4]:
# Check for duplicates
duplicate_rows = metro.duplicated().sum()
duplicate_timestamps = metro['timestamp'].duplicated().sum()

print(f"Duplicate rows: {duplicate_rows}")
print(f"Duplicate timestamps: {duplicate_timestamps}")

Duplicate rows: 0
Duplicate timestamps: 0


### Findings

After initial cleaning, the dataset contains **1,516,948 records** and **16 columns** with no missing values. No duplicate rows or timestamps were found, confirming the data is clean and suitable for time-based analysis.

### Sort by timestamp and check date range

This will tell us exactly what time period the dataset covers:

In [5]:
metro = metro.sort_values('timestamp').reset_index(drop=True)

print(f"Date range: {metro['timestamp'].min()} → {metro['timestamp'].max()}")
print(f"Total duration: {metro['timestamp'].max() - metro['timestamp'].min()}")

Date range: 2020-02-01 00:00:00 → 2020-09-01 03:59:50
Total duration: 213 days 03:59:50


In [6]:
metro['time_diff'] = metro['timestamp'].diff()

print("Most common intervals:")
print(metro['time_diff'].value_counts().head(10))

print("\nInterval statistics:")
print(metro['time_diff'].describe())

Most common intervals:
time_diff
0 days 00:00:10    1337521
0 days 00:00:09     128277
0 days 00:00:12      38321
0 days 00:00:13       7988
0 days 00:00:11       4471
0 days 00:00:21         10
0 days 00:00:19          5
0 days 00:00:22          4
0 days 00:00:14          3
0 days 00:00:17          3
Name: count, dtype: int64

Interval statistics:
count                      1516947
mean     0 days 00:00:12.141221809
std      0 days 00:05:14.107266185
min                0 days 00:00:08
25%                0 days 00:00:10
50%                0 days 00:00:10
75%                0 days 00:00:10
max                2 days 00:01:58
Name: time_diff, dtype: object


### Findings

The temporal structure of the MetroPT dataset is close to regular, with a dominant sampling interval of 10 seconds and a median interval of 10 seconds as well. Small variations of 9–13 seconds are also present, which suggests minor timing irregularities typical of real-world sensor systems. In addition, the dataset contains a few much larger gaps, including interruptions of up to two days, which indicates that the recording is not perfectly continuous over the full monitoring period.

Checking largest gaps:

In [7]:
large_gaps = metro[metro['time_diff'] > pd.Timedelta(minutes=5)][['timestamp', 'time_diff']]
print(f"Number of gaps > 5 minutes: {len(large_gaps)}")
print("\nLargest gaps:")
print(large_gaps.nlargest(10, 'time_diff'))

Number of gaps > 5 minutes: 275

Largest gaps:
                  timestamp       time_diff
618538  2020-04-27 01:12:49 2 days 00:01:58
1055664 2020-06-28 23:07:43 1 days 12:14:36
214850  2020-03-01 04:00:09 1 days 04:03:01
1318888 2020-08-05 08:23:01 1 days 00:40:33
805415  2020-05-25 01:14:14 1 days 00:34:51
1126187 2020-07-08 15:20:51 0 days 23:56:00
1454438 2020-08-23 18:51:01 0 days 23:39:17
708881  2020-05-10 22:48:58 0 days 22:17:41
908125  2020-06-08 11:48:04 0 days 21:28:25
450114  2020-04-02 09:59:17 0 days 20:43:37


In [8]:
metro = metro.drop(columns=['time_diff'])

A total of 275 gaps longer than 5 minutes were detected. 
The presence of larger temporal gaps may indicate periods when the monitored system was offline, under maintenance, or temporarily removed from operation. This interpretation is consistent with the run-to-failure character of the dataset, although the exact cause of each interruption cannot be confirmed from the sensor table alone.

### Continuous vs Binary Columns

The documentation confirms that several 0/1 variables in MetroPT are not ordinary sensor measurements, but electrical or control signals describing machine state, alarms, and operating conditions. This supports treating them separately from continuous pressure, temperature, and current measurements during the analysis.

In [9]:
binary_cols = [
    col for col in metro.columns
    if col != 'timestamp' and metro[col].nunique(dropna=True) == 2
]

continuous_cols = [
    col for col in metro.columns
    if col != 'timestamp' and col not in binary_cols
]

print(f"Binary/state columns ({len(binary_cols)}): {binary_cols}")
print(f"Continuous columns ({len(continuous_cols)}): {continuous_cols}")

Binary/state columns (8): ['COMP', 'DV_eletric', 'Towers', 'MPG', 'LPS', 'Pressure_switch', 'Oil_level', 'Caudal_impulses']
Continuous columns (7): ['TP2', 'TP3', 'H1', 'DV_pressure', 'Reservoirs', 'Oil_temperature', 'Motor_current']


In [10]:
for col in binary_cols:
    print(f"{col}: {sorted(metro[col].unique())}")

COMP: [np.float64(0.0), np.float64(1.0)]
DV_eletric: [np.float64(0.0), np.float64(1.0)]
Towers: [np.float64(0.0), np.float64(1.0)]
MPG: [np.float64(0.0), np.float64(1.0)]
LPS: [np.float64(0.0), np.float64(1.0)]
Pressure_switch: [np.float64(0.0), np.float64(1.0)]
Oil_level: [np.float64(0.0), np.float64(1.0)]
Caudal_impulses: [np.float64(0.0), np.float64(1.0)]


In [11]:
binary_cols = [
    'COMP', 'DV_eletric', 'Towers', 'MPG',
    'LPS', 'Pressure_switch', 'Oil_level', 'Caudal_impulses'
]

continuous_cols = [
    'TP2', 'TP3', 'H1', 'DV_pressure',
    'Reservoirs', 'Oil_temperature', 'Motor_current'
]

### Findings

The MetroPT variables were separated into continuous sensor measurements and binary-valued state-like variables. Automatic detection based on the number of unique values identified eight binary-valued columns, all of which were confirmed to contain only 0/1 values.

In [12]:
# continuous summary

metro[continuous_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
TP2,1516948.0,1.367826,3.250930,-0.032,-0.014,-0.012,-0.0100,10.676
TP3,1516948.0,8.984611,0.639095,0.730,8.492,8.960,9.4920,10.302
H1,1516948.0,7.568155,3.333200,-0.036,8.254,8.784,9.3740,10.288
DV_pressure,1516948.0,0.055956,0.382402,-0.032,-0.022,-0.020,-0.0180,9.844
Reservoirs,1516948.0,8.985233,0.638307,0.712,8.494,8.960,9.4920,10.300
Oil_temperature,1516948.0,62.644182,6.516261,15.400,57.775,62.700,67.2500,89.050
Motor_current,1516948.0,2.050171,2.302053,0.020,0.040,0.045,3.8075,9.295


In [13]:
# binary frequency tables

binary_summary = pd.DataFrame({
    col: metro[col].value_counts(normalize=True)
    for col in binary_cols
}).T.fillna(0)

In [14]:
binary_summary

,0.0,1.0
COMP,0.163043,0.836957
DV_eletric,0.839389,0.160611
Towers,0.080152,0.919848
MPG,0.167336,0.832664
LPS,0.996580,0.003420
Pressure_switch,0.008563,0.991437
Oil_level,0.095844,0.904156
Caudal_impulses,0.062893,0.937107


The binary-valued variables are strongly imbalanced, with several state indicators spending most of the time in a single dominant state. Notably, COMP=1 (83.7%) indicates the compressor is OFF or offloaded, not actively compressing, suggesting it operates in short active bursts. LPS (0.34% active) and Pressure_switch (0.86% active) function as alarm signals, triggering only under abnormal conditions such as pressure drops - making them particularly 
relevant for pre-failure detection.

In [15]:
metro.to_csv('../data/processed/metro_cleaned.csv', index=False)
print("Saved successfully!")

Saved successfully!


In [16]:
# Verify saved file
metro_check = pd.read_csv('../data/processed/metro_cleaned.csv')
print(metro_check.shape)
print(metro_check.columns.tolist())
print(metro_check.head(2))

(1516948, 16)
['timestamp', 'TP2', 'TP3', 'H1', 'DV_pressure', 'Reservoirs', 'Oil_temperature', 'Motor_current', 'COMP', 'DV_eletric', 'Towers', 'MPG', 'LPS', 'Pressure_switch', 'Oil_level', 'Caudal_impulses']
             timestamp    TP2    TP3     H1  DV_pressure  Reservoirs  \
0  2020-02-01 00:00:00 -0.012  9.358  9.340       -0.024       9.358   
1  2020-02-01 00:00:10 -0.014  9.348  9.332       -0.022       9.348   

   Oil_temperature  Motor_current  COMP  DV_eletric  Towers  MPG  LPS  \
0           53.600           0.04   1.0         0.0     1.0  1.0  0.0   
1           53.675           0.04   1.0         0.0     1.0  1.0  0.0   

   Pressure_switch  Oil_level  Caudal_impulses  
0              1.0        1.0              1.0  
1              1.0        1.0              1.0  
